# AE_export_frozen_model: the frozen model applied in PIC

Builds `PIC_SUB_02_frozen_wuhan_model.json`, the file that the PIC notebooks apply without refitting, from the saved state of a
full run of `AE_subphenotypes_analysis.ipynb`: the six-class model (weights, means, variances), the preprocessing constants, the
effective reference limits by age band, the CRP reporting floor, the Fleming centiles, the S1 bedside rule with its apparent
performance, and aggregate reference values of the derivation cohort. It also checks that the file reproduces the reported
posterior probabilities and how the external preprocessing assigns the derivation children.

**Where to run it.** In the folder of the full run (it reads `output_subphenotypes/code/ae_lpa_tools.py` and
`cache_figure_state/figure_state_INTERNAL.pkl`, which hold child-level data and stay on that computer). Run All.

**Outputs** in `output_frozen_model/`, aggregate only: `PIC_SUB_02_frozen_wuhan_model.json`, `export_checks.json` and
`effective_reference_limits.csv`. The last cell compares the model file with the one deposited on OSF and kept in
`pic_external_validation/`.

In [ ]:
# Setup: inputs of the full run and the output folder.
import hashlib
import json
import pickle
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.special import logsumexp
from scipy.stats import beta
from sklearn.metrics import cohen_kappa_score

RUN = Path.cwd()                                                # the folder in which AE_subphenotypes_analysis.ipynb ran
for p in (RUN / 'output_subphenotypes' / 'code' / 'ae_lpa_tools.py', RUN / 'cache_figure_state' / 'figure_state_INTERNAL.pkl'):
    if not p.is_file():
        raise FileNotFoundError(f'{p} not found: run this notebook in the folder of a full run of AE_subphenotypes_analysis.ipynb')
sys.path.insert(0, str(RUN / 'output_subphenotypes' / 'code'))
import ae_lpa_tools as T  # noqa: E402

with open(RUN / 'cache_figure_state' / 'figure_state_INTERNAL.pkl', 'rb') as f:
    S = pickle.load(f)
O = S['objects']
OUT = RUN / 'output_frozen_model'
OUT.mkdir(exist_ok=True)

RAW, XC, CLU, FINAL = O['RAW'], O['XC'], O['CLU'], O['FINAL']
LOG_SCALE, LOWER_LIMIT, LABEL = O['LOG_SCALE'], O['LOWER_LIMIT'], O['LABEL']
LAB_IND = O['LAB_IND']
D, L = O['D'], O['LABS_ALL']
W, MU, VAR = O['MAIN_TUPLE']
POST, MODAL, CLS = O['POST'], np.asarray(O['MODAL']), O['CLS']
EARLY = np.asarray(O['EARLY'])
assert list(XC.columns) == FINAL and len(FINAL) == 24 and len(CLU) == 695

In [ ]:
# 1. preprocessing constants (log, winsorizing, standardization)
pre = {}
X = RAW.loc[CLU, FINAL].astype(float).copy()
for c in FINAL:
    x = np.log(X[c]) if c in LOG_SCALE else X[c]
    lo, hi = float(x.quantile(0.005)), float(x.quantile(0.995))
    xw = x.clip(lo, hi)
    pre[c] = {'log': c in LOG_SCALE, 'winsor_low': lo, 'winsor_high': hi, 'mean': float(xw.mean()), 'sd': float(xw.std(ddof=1))}


def apply_pre(raw, cols):
    Z = pd.DataFrame(index=raw.index)
    for c in cols:
        p = pre[c]
        x = raw[c].astype(float)
        x = np.log(x) if p['log'] else x
        Z[c] = (x.clip(p['winsor_low'], p['winsor_high']) - p['mean']) / p['sd']
    return Z


Z = apply_pre(RAW.loc[CLU], FINAL)
chk_pre = float(np.nanmax(np.abs(Z.values - XC.values)))
same_nan = bool((np.isnan(Z.values) == np.isnan(XC.values)).all())
post_re, _ = T.posterior(Z.values, (W, MU, VAR))
chk_post = float(np.max(np.abs(post_re - POST)))
chk_modal = int((post_re.argmax(1) == MODAL).sum())
print(f'preprocessing max diff {chk_pre:.2e}, NaN pattern equal {same_nan}; posterior max diff {chk_post:.2e}; modal agree {chk_modal}/695')
assert chk_pre < 1e-10 and same_nan and chk_post < 1e-10 and chk_modal == 695

In [ ]:
# 2. effective reference limits by age band
# Age bands: the ages at which the age-specific limits attached to the Wuhan results change (6 months; 1, 2, 6 and 13 years).
BANDS = [('28d-<6mo', 28, 182.625), ('6mo-<1y', 182.625, 365.25), ('1-<2y', 365.25, 730.5), ('2-<6y', 730.5, 2191.5),
         ('6-<13y', 2191.5, 4748.25), ('13-<18y', 4748.25, 6574.5)]


def band_of(age_days):
    for name, lo, hi in BANDS:
        if lo <= age_days < hi:
            return name
    return None


first = (L[L.analyte.isin(LAB_IND) & L.h.between(-24, 24)]
         .drop_duplicates(['linked_admission_id', 'analyte'], keep='first'))
first = first[first.linked_admission_id.isin(set(CLU))].copy()
first['band'] = first.linked_admission_id.map(D.age_days).map(band_of)
assert first.band.notna().all()
MIN_CELL = 10


def eff(limits, log):
    """Geometric mean (log-scale tests) or harmonic mean (linear-scale tests) of the limits: a value divided by it equals, on the
    model's scale, the average of that value divided by the limits attached to the Wuhan results."""
    v = np.asarray(limits, float)
    return float(np.exp(np.mean(np.log(v)))) if log else float(1 / np.mean(1 / v))


REF, ref_rows = {}, []
for a in LAB_IND:
    s = first[first.analyte.eq(a)]
    log = a in LOG_SCALE
    overall = eff(s.limit, log)
    REF[a] = {'side': 'lower' if a in LOWER_LIMIT else 'upper', 'unit': s.unit.mode().iloc[0], 'all_ages': overall, 'bands': {}}
    for name, _, _ in BANDS:
        sb = s[s.band.eq(name)]
        val, src = (eff(sb.limit, log), 'band') if len(sb) >= MIN_CELL else (overall, 'all ages')
        REF[a]['bands'][name] = {'limit': val, 'results': int(len(sb)), 'source': src}
        ref_rows.append({'analyte': a, 'band': name, 'results': len(sb), 'effective_limit': round(val, 4), 'source': src,
                         'distinct_limits': int(sb.limit.nunique())})
REF_TAB = pd.DataFrame(ref_rows)


def limit_for(a, age_days):
    return REF[a]['bands'][band_of(age_days)]['limit']

In [ ]:
# 2b. Wuhan lower reporting limits (left-censored first values)
# A test reported as below a limit in at least 5% of the Wuhan first values has a point mass at that limit on the model's scale
# (hs-CRP: the least abnormal class S6 sits there). External results below that limit, or reported as below any limit, then enter at
# Wuhan's limit (the median of the censored values divided by their reference limits).
FLOOR_MIN_PCT = 5.0
FLOORS = {}
for a in LAB_IND:
    s = first[first.analyte.eq(a)]
    c = s[s.censor.eq('<')]
    pct_c = 100 * len(c) / len(s) if len(s) else 0.0
    FLOORS[a] = {'censored_low_n': int(len(c)), 'censored_low_pct': round(pct_c, 1),
                 'floor_ratio': float(np.median(c.value / c.limit)) if len(c) else None,
                 'floor_value_median': float(c.value.median()) if len(c) else None, 'applied': bool(pct_c >= FLOOR_MIN_PCT)}
print('floors applied:', {a: f for a, f in FLOORS.items() if f['applied']})
_cls = pd.Series(MODAL, index=CLU)
_crp = first[first.analyte.eq('hs_crp')]
FLOORS['hs_crp']['censored_pct_by_class'] = {CLS[k]: round(100 * float(_crp[_crp.linked_admission_id.map(_cls).eq(k)].censor.eq('<').mean()), 1)
                                             for k in range(6)}

In [ ]:
# 3. the external-data preprocessing applied to the Wuhan children
VAL = O['LABVAL'].loc[CLU]          # first values in the Wuhan unit, 22 tests
ages = D.loc[CLU, 'age_days'].values
RAW_E = RAW.loc[CLU].copy()
for a in LAB_IND:
    RAW_E[a] = [v / limit_for(a, ad) if pd.notna(v) else np.nan for v, ad in zip(VAL[a].values, ages)]
# the reporting floor, as applied to external data (hs-CRP): censored-low first values, or ratios below the floor, enter at it
_cen = first.pivot(index='linked_admission_id', columns='analyte', values='censor').reindex(index=CLU)
for a, f in FLOORS.items():
    if f['applied']:
        low = (_cen[a].eq('<').values if a in _cen else False) | (RAW_E[a].values < f['floor_ratio'])
        RAW_E.loc[low & RAW_E[a].notna().values, a] = f['floor_ratio']
post_e, _ = T.posterior(apply_pre(RAW_E, FINAL).values, (W, MU, VAR))
modal_e = post_e.argmax(1)
kap_e = float(cohen_kappa_score(MODAL, modal_e))
agree_e = float((MODAL == modal_e).mean())
print(f'External preprocessing (age-band limits and hs-CRP floor) applied to Wuhan: agreement {agree_e:.3f}, kappa {kap_e:.3f}; '
      f'S1 kept {int(((MODAL == 0) & (modal_e == 0)).sum())}/53, added {int(((MODAL != 0) & (modal_e == 0)).sum())}')

In [ ]:
# 4. Wuhan reference values (aggregate, modal assignment)
Dc = D.loc[CLU].copy()
Dc['cls'] = [CLS[m] for m in MODAL]
picu = Dc[Dc.any_picu_stay.astype(bool)]
ref = {'children': 695, 'class_n': Dc.cls.value_counts().reindex(CLS).astype(int).to_dict(),
       'picu_children': int(len(picu)), 'picu_class_n': picu.cls.value_counts().reindex(CLS).fillna(0).astype(int).to_dict(),
       'S1_pct': round(100 * float((Dc.cls == 'S1').mean()), 1), 'S1_pct_picu': round(100 * float((picu.cls == 'S1').mean()), 1),
       'age_years_median_iqr': [round(float(x), 2) for x in Dc.age_years.quantile([.5, .25, .75])]}
for col, key in [('death', 'in_hospital_death'), ('death_or_dama_critical_final', 'death_or_critical_discharge')]:
    y = Dc[col].astype(float)
    ref[key] = {'events_by_class': {c: int(y[Dc.cls == c].sum()) for c in CLS},
                'risk_pct_by_class': {c: round(100 * float(y[Dc.cls == c].mean()), 1) for c in CLS},
                'rr_S1_vs_others': round(float(y[Dc.cls == 'S1'].mean() / y[Dc.cls != 'S1'].mean()), 2)}
ref['validators'] = {}
for v in ['il6', 'il10', 'tnf_alpha', 'ifn_gamma', 'ferritin']:
    lv = Dc['log_' + v]
    gm = {c: float(np.exp(lv[Dc.cls == c].mean())) for c in CLS}
    ref['validators'][v] = {'geometric_mean_by_class': {c: round(g, 3) for c, g in gm.items()},
                            'measured_by_class': {c: int(lv[Dc.cls == c].notna().sum()) for c in CLS},
                            'ratio_S1_vs_S6': round(gm['S1'] / gm['S6'], 2),
                            'ratio_S1_vs_others': round(float(np.exp(lv[Dc.cls == 'S1'].mean() - lv[Dc.cls != 'S1'].mean())), 2)}
ref['h2_reported_bch'] = {'il6_ratio_S1_vs_S6': [46.89, 15.95, 116.59], 'il10_ratio_S1_vs_S6': [18.45, 8.21, 34.24]}
X0, Mf, X02 = T.prep(XC.values)
lp = T._log_emission(X0, Mf, X02, MU, VAR) + np.log(W)[None, :]
llv = logsumexp(lp, axis=1) / Mf.sum(1)
mx = POST.max(1)
ref['fit'] = {'loglik_per_value_mean': round(float(llv.mean()), 4), 'loglik_per_value_median': round(float(np.median(llv)), 4),
              'loglik_per_value_p05': round(float(np.percentile(llv, 5)), 4), 'max_posterior_mean': round(float(mx.mean()), 4),
              'max_posterior_ge_0_8_pct': round(100 * float((mx >= 0.8).mean()), 1), 'missing_pct': round(100 * float(np.isnan(XC.values).mean()), 1)}
r_between = np.corrcoef(MU)
ref['profile_r_S1_vs_other_wuhan_classes'] = {CLS[k]: round(float(r_between[0, k]), 3) for k in range(1, 6)}


def dist(s):
    s = pd.Series(s, dtype=float).dropna()
    return {'pct_measured': None, 'median': round(float(s.median()), 4), 'p25': round(float(s.quantile(.25)), 4),
            'p75': round(float(s.quantile(.75)), 4)}


ref['indicators'] = {}
for a in LAB_IND:
    d = dist(VAL[a])
    d['pct_measured'] = round(100 * float(VAL[a].notna().mean()), 1)
    d['unit'] = REF[a]['unit']
    ref['indicators'][a] = d
for v, col in [('temp_c', 'temp_c'), ('heart_rate', 'heart_rate'), ('resp_rate', 'resp_rate')]:
    s = O['VITALS_RAW'].loc[CLU, col]
    d = dist(s)
    d['pct_measured'] = round(100 * float(s.notna().mean()), 1)
    ref['indicators'][v] = d
VALID = O['VALID'].loc[CLU]
for v in ['il6', 'il10', 'tnf_alpha', 'ifn_gamma', 'ferritin']:
    d = dist(VALID[v])
    d['pct_measured'] = round(100 * float(VALID[v].notna().mean()), 1)
    ref['indicators'][v] = d

In [ ]:
# 5. S1 bedside rule (derived on these Wuhan children; exploratory)
RULE = {'name': 'S1 bedside rule', 'criteria': [{'test': 'ast', 'op': '>=', 'value': 200, 'unit': 'U/L'},
                                                {'test': 'ldh', 'op': '>=', 'value': 700, 'unit': 'U/L'},
                                                {'test': 'inr', 'op': '>=', 'value': 1.5, 'unit': 'ratio'}],
        'positive_if_at_least': 2, 'assessable_if': 'AST and LDH both measured (a test not done counts as not met)',
        'window': 'first value from 24 h before to 24 h after hospital admission'}


def rule_pos(V):
    c = np.vstack([(V['ast'] >= 200).values, (V['ldh'] >= 700).values, (V['inr'] >= 1.5).values])
    return c.sum(axis=0) >= 2


def cp(k, n):
    return [round(float(beta.ppf(0.025, k, n - k + 1)) if k > 0 else 0.0, 3), round(float(beta.ppf(0.975, k + 1, n - k)) if k < n else 1.0, 3)]


pos, s1 = rule_pos(VAL), MODAL == 0
assess = (VAL.ast.notna() & VAL.ldh.notna()).values
perf = {}
for nm, m in [('all', assess), ('2019-2022', assess & EARLY), ('2023-2026', assess & ~EARLY)]:
    p, t = pos[m], s1[m]
    tp, fn, fp, tn = int((p & t).sum()), int((~p & t).sum()), int((p & ~t).sum()), int((~p & ~t).sum())
    perf[nm] = {'assessable': int(m.sum()), 'tp': tp, 'fn': fn, 'fp': fp, 'tn': tn,
                'sensitivity': [round(tp / (tp + fn), 3)] + cp(tp, tp + fn), 'specificity': [round(tn / (tn + fp), 3)] + cp(tn, tn + fp),
                'ppv': [round(tp / (tp + fp), 3)] + cp(tp, tp + fp), 'npv': [round(tn / (tn + fn), 3)] + cp(tn, tn + fn)}
for col, key in [('death_or_dama_critical_final', 'death_or_critical_discharge'), ('death', 'in_hospital_death')]:
    o = Dc[col].astype(bool).values
    a_, b_, c_, d_ = int(o[pos & assess].sum()), int((pos & assess).sum()), int(o[~pos & assess].sum()), int((~pos & assess).sum())
    rr = (a_ / b_) / (c_ / d_)
    se = np.sqrt(1 / a_ - 1 / b_ + 1 / c_ - 1 / d_)
    perf[key] = {'rule_positive_events': a_, 'rule_positive': b_, 'rule_negative_events': c_, 'rule_negative': d_,
                 'risk_ratio': [round(rr, 2), round(rr * np.exp(-1.96 * se), 2), round(rr * np.exp(1.96 * se), 2)]}
RULE['wuhan_performance'] = perf
RULE['derivation'] = ('Derived on the 695 Wuhan children against the model\'s S1 assignment: single tests and counts of 2 to 5 criteria '
                      'with clinically round thresholds, among tests measured in at least 80% of children in both Wuhan and PIC '
                      '(procalcitonin and D-dimer excluded); chosen by the highest Youden index, ties broken by fewer tests. Exploratory.')
print(json.dumps(perf['all'], indent=None))

In [ ]:
# 6. Fleming centiles (copied from the analysis notebook)
FLEMING = {'source': 'Fleming S et al. Lancet 2011;377:1011-8, web appendix Web Tables 4 (respiratory rate) and 5 (heart rate)',
           'bands_months': O['FLEMING_BANDS_MONTHS'], 'centiles': [1, 10, 25, 50, 75, 90, 99],
           'heart_rate': O['FLEMING_HR'], 'resp_rate': O['FLEMING_RR'],
           'method': 'piecewise-linear interpolation between centiles on the standard-normal scale; linear extrapolation beyond the '
                     '1st and 99th centiles with the slope of the outer interval; age in months = age in days / 30.4375'}

In [ ]:
# 7. write the package
package = {
    'name': 'Frozen Wuhan admission subphenotype model for external application (reported run of 14 Oct 2026, AE_freeze_v1.4)',
    'version': 1,
    'classes': CLS, 'class_labels': {'S1': 'hyperinflammatory', 'S6': 'least abnormal'},
    'indicators': FINAL, 'indicator_labels': {c: LABEL[c] for c in FINAL},
    'lab_tests_22': LAB_IND, 'model_children': 695,
    'weights': [float(x) for x in W], 'means': [[float(x) for x in row] for row in MU], 'variances': [[float(x) for x in row] for row in VAR],
    'preprocessing': pre, 'reference_limits': REF, 'reporting_floors': FLOORS,
    'age_bands': [{'name': n, 'from_days': lo, 'to_days': hi} for n, lo, hi in BANDS],
    'fleming': FLEMING, 'bedside_rule': RULE, 'wuhan_reference': ref,
    'checks_in_wuhan': {'reproduces_reported_posteriors_max_abs_difference': chk_post,
                        'external_preprocessing_applied_to_wuhan': {'agreement': round(agree_e, 4), 'kappa': round(kap_e, 4),
                                                                    'agreement_n': int((MODAL == modal_e).sum()), 'children': int(len(MODAL)),
                                                                    'S1_kept': int(((MODAL == 0) & (modal_e == 0)).sum()),
                                                                    'S1_reported': int((MODAL == 0).sum()),
                                                                    'S1_added': int(((MODAL != 0) & (modal_e == 0)).sum())}},
    'how_to_apply': ['Laboratory value in the Wuhan unit divided by reference_limits[test].bands[age band].limit (lower limit for '
                     'lymphocyte percentage, platelets, hemoglobin, albumin, fibrinogen and sodium; upper limit otherwise). The limits '
                     'are effective Wuhan limits: the geometric mean (log-scale tests) or harmonic mean (linear-scale tests) of the '
                     'limits attached to the first values of the 695 children in that age band (all ages where a band has fewer than '
                     '10 results).',
                     'Reporting floors: for a test with reporting_floors[test].applied, a result reported as below a limit, or whose '
                     'ratio to the reference limit is below floor_ratio, enters with ratio floor_ratio (hs-CRP only).',
                     'Heart and respiratory rate: z-score for age with the Fleming centiles.',
                     'Then natural log where preprocessing[indicator].log is true, clipping at winsor_low and winsor_high, and '
                     '(x - mean) / sd.',
                     'Posterior: diagonal Gaussian mixture with these weights, means and variances; missing indicators are left out '
                     'of the likelihood (full-information likelihood). Classes are in display order S1 (hyperinflammatory) to S6 '
                     '(least abnormal).'],
}


def canon(obj):
    return json.dumps(obj, sort_keys=True, ensure_ascii=True, separators=(',', ':'))


package['sha256_of_content'] = hashlib.sha256(canon(package).encode()).hexdigest()
path = OUT / 'PIC_SUB_02_frozen_wuhan_model.json'
path.write_text(json.dumps(package, indent=1, ensure_ascii=True), encoding='utf-8')
# round-trip check of the content hash
back = json.loads(path.read_text(encoding='utf-8'))
h = back.pop('sha256_of_content')
assert hashlib.sha256(canon(back).encode()).hexdigest() == h == package['sha256_of_content']
REF_TAB.to_csv(OUT / 'effective_reference_limits.csv', index=False)
checks = {'preprocessing_max_abs_difference': chk_pre, 'posterior_max_abs_difference': chk_post, 'modal_agreement': f'{chk_modal} of 695',
          'external_preprocessing_on_wuhan': package['checks_in_wuhan']['external_preprocessing_applied_to_wuhan'],
          'content_sha256': h, 'file_sha256': hashlib.sha256(path.read_bytes()).hexdigest()}
(OUT / 'export_checks.json').write_text(json.dumps(checks, indent=1), encoding='utf-8')
print(json.dumps(checks, indent=1))

In [ ]:
# Comparison with the deposited frozen model (OSF; pic_external_validation/PIC_SUB_02_frozen_wuhan_model.json)
DEPOSITED = {'content_sha256': 'd02b69ead00732b4b46b8bf67ff46dc46e5727eb43c735247b1d57badbb408f5',
             'file_sha256': 'a8a51287d7dd2cb67b623328ffedc663deb9a01b3c5364db3c4e9d8f39be2179'}
for k, v in DEPOSITED.items():
    print(f"{k:15s} {'MATCH' if checks[k] == v else 'DIFFERENT'}  {checks[k][:16]}")